In [0]:
%sql
-- ============================================================================
-- Proyecto Medallion - Catalogo "cobranzas"
-- Paso 2: DDL de silver.cuentas y silver.cuentas_rechazados (cuarentena)
-- ============================================================================

USE CATALOG cobranzas;

-- ----------------------------------------------------------------------------
-- silver.cuentas
-- ----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS silver.cuentas (
  silver_key                          STRING NOT NULL COMMENT 'Llave subrogada: hash(numero_de_prestamo_vp + llave natural de la promesa)',

  -- datos de cuentas (bronze.cuentas)
  jefatura                            STRING,
  responsable                         STRING,
  gestor                              STRING,
  numero_de_prestamo_vp               STRING,
  numero_cliente_unico                STRING,
  dni                                  STRING,
  nombre                               STRING,
  saldo_l                             DECIMAL(18,2),
  saldo_usd                           DECIMAL(18,2),
  saldo_total_usd                     DECIMAL(18,2),
  saldo_total_honorarios              DECIMAL(18,2),
  fecha_asignacion                    DATE,
  estado_cartera                      STRING,
  producto                            STRING,
  segmentacion_producto               STRING,
  tipo_producto                       STRING,
  fecha_castigo                       DATE,
  fecha_ultimo_pago                   DATE,
  base_laboral                        STRING,
  tipo_empresa                        STRING,
  lugar_trabajo                       STRING,
  bienes_inmuebles                    STRING,
  email                               STRING,
  telefonos                           STRING,
  fecha_ultima_visita                 DATE,
  celular                             STRING,
  gestion_historica                   STRING,
  gestion_diaria                      STRING,
  fecha_ultima_gestion                DATE,
  caracterizacion_del_mes             STRING,
  codigos_gestion                     STRING,
  sub_caracterizacion                 STRING,
  monto_promesa                       DECIMAL(18,2),
  fecha_promesa                       DATE,
  fecha_solicitud_documento           DATE,
  fecha_recepcion_documentos          DATE,
  fecha_presentacion_demanda          DATE,
  cuantia_demandada_lps               DECIMAL(18,2),
  cuantia_demandada_usd               DECIMAL(18,2),
  tipo_demanda                        STRING,
  estado_demanda                      STRING,
  caracterizacion_judicial            STRING,
  fecha_ultimo_movimiento_judicial    DATE,
  gestion_judicial_historica          STRING,
  fecha_ultima_revision_expediente    DATE,
  fecha_requerimiento                 DATE,
  fecha_embargo                       DATE,
  embargo_de                          STRING,
  cantidad_mensual_retenida           DECIMAL(18,2),
  juzgado                             STRING,
  expediente                          STRING,
  juez                                STRING,
  ciudad_demanda                      STRING,
  ciudad                              STRING,

  -- datos de la promesa (bronze.promesas), null si la cuenta no tiene promesa
  monto_mensual_lps_promesa           DECIMAL(18,2),
  monto_mensual_dolar_promesa         DECIMAL(18,2),
  fecha_de_pago_promesa               DATE,
  tipo_arreglo_promesa                STRING,
  meses_promesa                       INT,
  fecha_inicio_promesa                DATE,
  fecha_final_promesa                 DATE,
  estado_promesa                      STRING,

  -- auditoria
  bronze_source_file_cuenta           STRING,
  bronze_source_file_promesa          STRING,
  silver_processed_at                 TIMESTAMP
)
COMMENT 'Silver: cuentas combinadas con sus promesas de pago (1 fila por cuenta+promesa), solo registros validos'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);


-- ----------------------------------------------------------------------------
-- silver.cuentas_rechazados (cuarentena)
-- ----------------------------------------------------------------------------
-- Mismas columnas que silver.cuentas pero TODO en STRING (se guarda el dato
-- crudo, ya renombrado y normalizado, tal como llego intentando castear -
-- para que puedan revisar exactamente que vino mal) + el detalle de errores.

CREATE TABLE IF NOT EXISTS silver.cuentas_rechazados (
  silver_key                          STRING,

  jefatura                            STRING,
  responsable                         STRING,
  gestor                              STRING,
  numero_de_prestamo_vp               STRING,
  numero_cliente_unico                STRING,
  dni                                  STRING,
  nombre                               STRING,
  saldo_l                             STRING,
  saldo_usd                           STRING,
  saldo_total_usd                     STRING,
  saldo_total_honorarios              STRING,
  fecha_asignacion                    STRING,
  estado_cartera                      STRING,
  producto                            STRING,
  segmentacion_producto               STRING,
  tipo_producto                       STRING,
  fecha_castigo                       STRING,
  fecha_ultimo_pago                   STRING,
  base_laboral                        STRING,
  tipo_empresa                        STRING,
  lugar_trabajo                       STRING,
  bienes_inmuebles                    STRING,
  email                               STRING,
  telefonos                           STRING,
  fecha_ultima_visita                 STRING,
  celular                             STRING,
  gestion_historica                   STRING,
  gestion_diaria                      STRING,
  fecha_ultima_gestion                STRING,
  caracterizacion_del_mes             STRING,
  codigos_gestion                     STRING,
  sub_caracterizacion                 STRING,
  monto_promesa                       STRING,
  fecha_promesa                       STRING,
  fecha_solicitud_documento           STRING,
  fecha_recepcion_documentos          STRING,
  fecha_presentacion_demanda          STRING,
  cuantia_demandada_lps               STRING,
  cuantia_demandada_usd               STRING,
  tipo_demanda                        STRING,
  estado_demanda                      STRING,
  caracterizacion_judicial            STRING,
  fecha_ultimo_movimiento_judicial    STRING,
  gestion_judicial_historica          STRING,
  fecha_ultima_revision_expediente    STRING,
  fecha_requerimiento                 STRING,
  fecha_embargo                       STRING,
  embargo_de                          STRING,
  cantidad_mensual_retenida           STRING,
  juzgado                             STRING,
  expediente                          STRING,
  juez                                STRING,
  ciudad_demanda                      STRING,
  ciudad                              STRING,

  monto_mensual_lps_promesa           STRING,
  monto_mensual_dolar_promesa         STRING,
  fecha_de_pago_promesa               STRING,
  tipo_arreglo_promesa                STRING,
  meses_promesa                       STRING,
  fecha_inicio_promesa                STRING,
  fecha_final_promesa                 STRING,
  estado_promesa                      STRING,

  _dq_errors                          ARRAY<STRING>,

  bronze_source_file_cuenta           STRING,
  bronze_source_file_promesa          STRING,
  silver_processed_at                 TIMESTAMP
)
COMMENT 'Cuarentena: filas cuenta+promesa que fallaron alguna validacion de COLUMN_CONFIG, con el detalle de errores en _dq_errors'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
